# Grouped-Query Attention (GQA) — a toy-scale build

A minimal implementation of **Grouped-Query Attention**, from Ainslie,
Lee-Thorp, de Jong, Zemlyanskiy, Lebrón, Sanghai, *"GQA: Training
Generalized Multi-Query Transformer Models from Multi-Head Checkpoints"*
(2023) — the simple, widely-deployed alternative to MLA (`../mla`) for
shrinking the KV cache.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The problem, again

Same motivation as MLA (`../mla`): the KV cache — everything a transformer
stores between tokens during generation — grows with `n_heads * d_head` per
token, and can dominate the memory cost of serving a model. MLA solves this
by compressing all heads' K/V into one small shared latent vector. GQA takes
a much simpler route.

## 2. The idea: fewer key/value heads than query heads

Ordinary **multi-head attention (MHA)** gives every query head its own
private key head and value head — `n_heads` of each. **Multi-query
attention (MQA)**, an earlier and more aggressive idea, goes to the other
extreme: every query head shares the *exact same single* key head and value
head. MQA shrinks the cache a lot, but can hurt quality — one shared K/V
pair has to serve every query head's needs at once.

**GQA sits in between.** Split the query heads into a small number of
*groups*; every head within a group shares one key head and one value head,
but different groups get different K/V heads:

```
n_kv_heads = n_heads // group_size
head h's key/value = kv_head[h // group_size]
```

- `n_kv_heads == n_heads` → this is just ordinary MHA (every group has size 1).
- `n_kv_heads == 1` → this is MQA (one giant group, everyone shares).
- Anything in between → GQA, trading off cache size against quality.

The KV cache only needs to store `n_kv_heads` sets of keys and values, so
picking `n_kv_heads` well below `n_heads` shrinks the cache by roughly that
ratio, at a much smaller implementation cost than MLA's latent compression.

## 3. Where this notebook fits in the "efficient attention" progression

This repo now has a small ladder of ideas for making attention's KV cache
cheaper:

```
MHA  ->  GQA  ->  MQA  ->  MLA
```

MHA is the baseline (this notebook, with `n_kv_heads = n_heads`). GQA and
MQA (this same notebook, with fewer KV heads) get there by literally
**sharing** K/V heads across groups of query heads. MLA (`../mla`) takes a
different, more involved approach — instead of sharing raw K/V heads,
it **compresses** all heads' content into one small latent vector and
reconstructs per-head keys/values from it on demand. Worth building both
and comparing: GQA is a few lines of code; MLA needs a whole compression +
decoupled-RoPE scheme.

> **Simplification used here:** none really needed — GQA is already about
> as simple as MHA itself. The one thing to notice in the code is
> `repeat_interleave`, which is what actually implements "every head in a
> group shares the same K/V head" by duplicating each KV head across its
> group before the attention computation.

In [ ]:
class GQA(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_kv_heads=2, d_head=16):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.h, self.hkv, self.dh = n_heads, n_kv_heads, d_head
        self.group_size = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * d_head, bias=False)   # smaller KV cache: fewer heads
        self.v_proj = nn.Linear(d_model, n_kv_heads * d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * d_head, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Hkv, Dh, G = self.h, self.hkv, self.dh, self.group_size
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)         # B,H,T,Dh
        k = self.k_proj(x).view(B, T, Hkv, Dh).transpose(1, 2)        # B,Hkv,T,Dh -- this is what actually gets cached
        v = self.v_proj(x).view(B, T, Hkv, Dh).transpose(1, 2)
        k = k.repeat_interleave(G, dim=1)                              # broadcast each KV head to its Q-head group
        v = v.repeat_interleave(G, dim=1)

        scores = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn = scores.masked_fill(mask, float('-inf')).softmax(-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(o)

## 4. Seeing MHA, GQA, and MQA as one family

Before assembling the full toy LM, a quick check that `n_kv_heads` really
does interpolate between MHA and MQA, and that gradients flow at every
setting.

In [ ]:
for n_kv in [4, 2, 1]:   # 4 -> ordinary MHA, 2 -> GQA, 1 -> MQA
    m = GQA(n_heads=4, n_kv_heads=n_kv)
    x = torch.randn(2, 10, 64, requires_grad=True)
    out = m(x)
    out.sum().backward()
    label = {4: "MHA", 2: "GQA", 1: "MQA"}[n_kv]
    print(f"n_kv_heads={n_kv} ({label}): out {tuple(out.shape)}, grads ok "
          f"{all(p.grad is not None for p in m.parameters())}")

## 5. Assembling a tiny language model

This notebook trains with `n_kv_heads=2` (out of 4 query heads) — real GQA,
not the MHA or MQA extremes.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([GQA(d_model, n_heads=4, n_kv_heads=2, d_head=16) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through GQA once it's wired into a real model. So the rest of this
notebook:

1. wraps GQA into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Measure the cache savings directly.** For a given `n_heads` and
  `d_head`, print the KV cache size (`n_kv_heads * d_head * 2` per token)
  at a few different `n_kv_heads` settings and see how it scales down as
  you approach MQA.
- **Compare against MLA** (`../mla`) on the same toy task — both solve the
  same problem, but GQA does it by *sharing* raw K/V heads while MLA does
  it by *compressing* into a latent and reconstructing. Worth timing both
  and comparing parameter counts.
- **Try training MHA and MQA versions** (the `n_kv_heads=4` and
  `n_kv_heads=1` extremes) on this toy task, and see whether the extreme
  KV-sharing of MQA costs anything on a task this simple — a real
  difference is more likely to show up on tasks that need more precise,
  per-head attention patterns.

Reference: Ainslie, Lee-Thorp, de Jong, Zemlyanskiy, Lebrón, Sanghai,
*"GQA: Training Generalized Multi-Query Transformer Models from Multi-Head
Checkpoints,"* 2023.